# 🌿 Hierarchical Clustering
**ISLP Chapter 12 · Data Pattern Recognition for the Rest of Us**

> Hierarchical clustering builds a tree of clusters — you can explore any number of segments by cutting the dendrogram at different heights. Unlike K-means, you don't need to specify K upfront.

### Dataset
**S&P 500 Stock Returns — Sector Clustering**
We cluster major US stocks by their daily return correlations. Stocks that move together get clustered together — revealing sector structure and diversification opportunities. This is how portfolio managers and risk analysts actually use clustering.

### Time: ~55 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
import scipy.stats as stats
print("\u2713 Setup complete")

In [ ]:
# S&P 500 representative stocks — synthetic returns with realistic sector correlations
# Stocks within the same sector have correlated returns; cross-sector correlation is lower
np.random.seed(42)
n_days = 252  # one trading year

stocks = {
    'Tech':       ['AAPL','MSFT','GOOGL','META','NVDA','AMZN'],
    'Finance':    ['JPM','BAC','GS','MS','WFC','C'],
    'Healthcare': ['JNJ','PFE','UNH','ABT','MRK','TMO'],
    'Energy':     ['XOM','CVX','COP','SLB','PSX','VLO'],
    'Consumer':   ['PG','KO','PEP','WMT','COST','MCD'],
}

all_tickers = [t for tickers in stocks.values() for t in tickers]
sector_labels = {t: s for s, tickers in stocks.items() for t in tickers}

# Market factor (all stocks move somewhat together)
market_factor = np.random.normal(0, 0.008, n_days)

# Generate correlated returns
returns_data = {}
for sector, tickers in stocks.items():
    # Sector factor — stocks in same sector move together
    sector_factor = np.random.normal(0, 0.006, n_days)
    for ticker in tickers:
        # Individual noise
        idio = np.random.normal(0, 0.012, n_days)
        # Energy has negative correlation with market during crises
        mkt_beta = -0.3 if sector == 'Energy' else 0.8
        returns_data[ticker] = mkt_beta*market_factor + 0.6*sector_factor + idio

returns = pd.DataFrame(returns_data, index=pd.date_range('2023-01-01', periods=n_days, freq='B'))

print("S&P 500 Representative Portfolio")
print(f"  {len(all_tickers)} stocks across {len(stocks)} sectors")
print(f"  {n_days} trading days of simulated returns")
print(f"\nSectors: {list(stocks.keys())}")
print(f"\nAverage daily return: {returns.mean().mean()*100:.3f}%")
print(f"Average daily volatility: {returns.std().mean()*100:.2f}%")

## 📐 Part 1 — Clustering Stocks by Return Correlation

Instead of clustering raw return values, we cluster by **correlation distance**:
```
distance(i,j) = 1 - correlation(i,j)
```
Stocks that move together (high correlation) have low distance and cluster together.
This is the standard approach in portfolio risk analysis.

In [ ]:
# Compute correlation matrix and convert to distance
corr_matrix = returns.corr()
dist_matrix = 1 - corr_matrix

# Hierarchical clustering — complete linkage
Z = linkage(squareform(dist_matrix.values), method='complete')

# Plot dendrogram
fig, ax = plt.subplots(figsize=(14, 5))
sector_colors = {'Tech':'#1e5fa8','Finance':'#e85d20','Healthcare':'#1a7a45',
                 'Energy':'#6b46c1','Consumer':'#0e7490'}
leaf_colors = [sector_colors[sector_labels[t]] for t in all_tickers]

dend = dendrogram(Z, labels=all_tickers, ax=ax, leaf_rotation=45,
                  color_threshold=0.7*max(Z[:,2]))
ax.set_title('Hierarchical Clustering of S&P 500 Stocks by Return Correlation\n'
             'Height = dissimilarity — stocks that merge early are most similar')
ax.set_ylabel('Correlation Distance (1 - correlation)')

# Add sector color legend
for sector, color in sector_colors.items():
    ax.plot([], [], 's', color=color, label=sector, markersize=8)
ax.legend(title='Actual Sector', loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()
print("\n\u2714 Stocks within the same sector cluster together at low height")
print("   Energy separates early (negative market correlation)")
print("   This confirms the sector structure without using any sector labels")

In [ ]:
# Cut dendrogram at different heights = different numbers of clusters
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, n_clusters in zip(axes, [3, 5, 7]):
    cluster_labels = fcluster(Z, n_clusters, criterion='maxclust')

    # Show correlation heatmap colored by cluster
    idx_sorted = np.argsort(cluster_labels)
    sorted_corr = corr_matrix.iloc[idx_sorted, idx_sorted]
    sorted_tickers = [all_tickers[i] for i in idx_sorted]

    im = ax.imshow(sorted_corr.values, cmap='RdYlGn', vmin=-0.5, vmax=1.0, aspect='auto')
    ax.set_xticks(range(len(sorted_tickers)))
    ax.set_yticks(range(len(sorted_tickers)))
    ax.set_xticklabels(sorted_tickers, rotation=90, fontsize=7)
    ax.set_yticklabels(sorted_tickers, fontsize=7)
    ax.set_title(f'{n_clusters} Clusters\n(sorted by cluster membership)')

    # Add cluster boundaries
    boundaries = np.where(np.diff(cluster_labels[idx_sorted]))[0] + 0.5
    for b in boundaries:
        ax.axhline(b, color='black', lw=1.5)
        ax.axvline(b, color='black', lw=1.5)

plt.colorbar(im, ax=axes[-1], label='Correlation', shrink=0.8)
plt.suptitle('Correlation Heatmaps at Different Cluster Cuts\n'
             'Green = high correlation, Red = negative correlation', fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Portfolio diversification insight
n_clusters = 5
cluster_labels = fcluster(Z, n_clusters, criterion='maxclust')
stock_clusters = pd.DataFrame({'Ticker': all_tickers,
                                'Sector': [sector_labels[t] for t in all_tickers],
                                'Cluster': cluster_labels})

print("=== Cluster Composition ===\n")
for cl in sorted(stock_clusters['Cluster'].unique()):
    members = stock_clusters[stock_clusters['Cluster']==cl]
    print(f"Cluster {cl} ({len(members)} stocks):")
    for _, row in members.iterrows():
        print(f"  {row['Ticker']:<8} [{row['Sector']}]")
    # Within-cluster avg correlation
    tickers = members['Ticker'].tolist()
    if len(tickers) > 1:
        avg_corr = corr_matrix.loc[tickers, tickers].values
        np.fill_diagonal(avg_corr, np.nan)
        print(f"  Avg within-cluster correlation: {np.nanmean(avg_corr):.3f}")
    print()

print("\u2714 Portfolio insight: pick ONE stock per cluster for maximum diversification")
print("   Stocks in the same cluster move together — owning multiple adds little diversification benefit")

In [ ]:
# @title 📝 Quiz — Hierarchical Clustering
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is the key advantage of hierarchical over K-means clustering?
# @markdown **Q2:** What does complete linkage measure when merging two clusters?
# @markdown **Q3:** Why do we cluster by correlation rather than raw return levels?
# @markdown **Q4:** How does a dendrogram help choose the number of clusters?
# @markdown **Q5:** If two stocks are in the same cluster, what does that mean for portfolio diversification?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit for AI Grading

In [ ]:
_NB_NAME_="Hierarchical Clustering"
# @title 📋 Get AI Feedback on Your Quiz
# @markdown **Step 1:** Enter your GitHub username below
# @markdown
# @markdown **Step 2:** Click ▶ Run — your answers are formatted as a prompt
# @markdown
# @markdown **Step 3:** Copy the output → click the **✨ Gemini button** in the Colab toolbar (top right) → paste → send
# @markdown
# @markdown **Step 4:** Keep the conversation going — ask Gemini to explain any output,
# @markdown code block, or chart from this notebook for deeper understanding

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    nd = sum(1 for v in answers.values() if str(v).strip())

    sep = "\u2550" * 62

    print(sep)
    print(f"  Student  : @{GITHUB_USERNAME.strip()}" if GITHUB_USERNAME.strip()
          else "  Student  : (set GITHUB_USERNAME above)")
    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answered : {nd}/{len(answers)} questions")
    print(sep)
    print()
    print("  \u2728  COPY THE PROMPT BELOW INTO GEMINI  \u2728")
    print("  (click the Gemini button in the Colab toolbar, top right)")
    print()
    print("\u2500" * 62)
    print()
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  - Say CORRECT, PARTIAL, or INCORRECT")
    print("  - One sentence: what I got right, or exactly what concept I should review")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - One specific thing to study if I struggled")
    print()
    print("After grading, I may paste specific code outputs or charts from this")
    print("notebook and ask follow-up questions — please help me understand them.")
    print()
    print("\u2500" * 62)
    print()
    print("  After pasting, keep the Gemini chat open and ask things like:")
    print("  \"Why did my SHAP waterfall plot show X?\"")
    print("  \"Can you explain what this output means: [paste output]\"")
    print("  \"I got a different result than the notebook — what went wrong?\"")
    print()
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork")
    print(sep)


---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*